# 🚀 CERN ROOT Data Analysis Workshop
Welcome to the interactive lab tutorial session! This notebook contains a comprehensive introduction to the ROOT framework, followed by your structured laboratory exercises and assignments.

## 🛠️ Step 1: Initialize the Lab Environment (Install Micromamba & ROOT Environment)
Run the following cell to install the necessary background tools.

* **Student Note:** Run this cell to build the background software environment containing CERN ROOT and Python libraries. This takes about 1–2 minutes on first run.

In [1]:
import sys, os, urllib.request, tarfile, subprocess

# 1. Install micromamba binary
micromamba_bin = "/usr/local/bin/micromamba"
if not os.path.exists(micromamba_bin):
    url = "https://micro.mamba.pm/api/micromamba/linux-64/latest"
    tar_path = "/tmp/micromamba.tar.bz2"
    urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path, "r:bz2") as tar:
        tar.extract("bin/micromamba", path="/usr/local", filter="data")

# 2. Detect notebook Python version and create root_env
py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"🔍 Creating root_env for Python {py_ver}...")
!/usr/local/bin/micromamba create -n root_env python={py_ver} root -c conda-forge -y

🔍 Creating root_env for Python 3.13...
[+] 0.0s
[+] 0.1s
conda-forge/linux-..  ⣾  
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                    ✔ Done (28.8 sec)

Resolving Environment                                                                     ✔ Done (1.1 sec)

Transaction

  Prefix: /root/.local/share/mamba/envs/root_env

  Updating specs:

   - python=3.13
   - root
   - pip


  Package                               Version  Build                  Channel          Size
───────────────────────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex                           4.5  7_kmp_llvm          

## 📦 Step 2: Fetch Workshop Materials (Clone Repository & Download Exercises)
* **Student Note:** Run this cell to clone the workshop GitHub repository containing the exercises, dataset text files, and tutorial macros.

In [2]:
import os, subprocess

repo_name = "root-data-analysis-workshop_-LHC_School_2026-"
repo_url = f"https://github.com/adeelurrahman/root-data-analysis-workshop_-LHC_School_2026-.git"

if not os.path.exists(repo_name):
    subprocess.run(["git", "clone", repo_url], check=True)
    print(f"✅ Cloned repository: {repo_name}")
else:
    print(f"📁 Repository directory '{repo_name}' already exists. Skipping clone.")

✅ Cloned repository: root-data-analysis-workshop_-LHC_School_2026-


## 🧩 Step 3: Load PyROOT & Initialize Environment (Configure Environment & Activate C++ Libraries)
* **Student Note:** Run this cell once per session to link CERN ROOT to Python and verify the PyROOT installation.


In [ ]:
import sys, os

env_prefix = os.path.expanduser("~/.local/share/mamba/envs/root_env")
conda_lib = os.path.join(env_prefix, "lib")
conda_libstdc = os.path.join(conda_lib, "libstdc++.so.6")

# 1. Force dynamic linker to load Conda's libstdc++ on process startup
if "LD_PRELOAD" not in os.environ or conda_libstdc not in os.environ["LD_PRELOAD"]:
    os.environ["LD_PRELOAD"] = conda_libstdc + ":" + os.environ.get("LD_PRELOAD", "")
    os.environ["LD_LIBRARY_PATH"] = conda_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")

    # Re-exec Python in-place so dynamic linker loads correct libstdc++ cleanly
    os.execv(sys.executable, [sys.executable] + sys.argv)

# 2. Configure Python search paths & environment variables
py_ver_str = f"python{sys.version_info.major}.{sys.version_info.minor}"
site_packages_path = os.path.join(env_prefix, "lib", py_ver_str, "site-packages")

if site_packages_path not in sys.path:
    sys.path.insert(0, site_packages_path)

os.environ["ROOTSYS"] = env_prefix
os.environ["PATH"] = f"{env_prefix}/bin:" + os.environ.get("PATH", "")

# 3. Import PyROOT
import ROOT
print(f"✅ CERN ROOT successfully loaded! Version: {ROOT.gROOT.GetVersion()}")

---
# 📖 Part 1: Core Framework Overview

### What is ROOT?
**ROOT is an open-source data analysis framework created by CERN**. It handles massive amounts of high-energy physics data, providing fast input/output, statistical analysis, curve fitting, and interactive graphics for complex scientific research.

### What is a `.root` File?
* A `.root` file is the standard data storage format used by the ROOT framework.
* It stores large, complex hierarchical data in a compressed binary format.
* It allows direct access to specific subsets of data without reading the entire file into memory.

### What Does a `.root` File Contain?
* **`TTree`**: A columnar dataset structure similar to a relational database or a data frame, optimized for fast reading.
* **`TBranch`**: Subdivisions of a `TTree` that group related variables or objects together.
* **Histograms (`TH1F`, `TH2F`)**: Frequency distributions used to plot data values.
* **Graphs (`TGraph`)**: XY data points with error bars.
* **Custom C++ Objects**: Serialized instances of user-defined classes.

### Basic Structure of the ROOT Library
* **Core**: Base classes, memory management, and system interfaces.
* **IO**: Classes for reading and writing data to disk (`TFile`, `TKey`).
* **Hist & Matrix**: Statistical containers, histograms, and linear algebra tools.
* **Tree**: Data storage and manipulation tools (`TTree`, `TChain`).
* **Graf & Gpad**: 2D and 3D graphics rendering engines.
* **MathCore & RooFit**: Advanced mathematical modeling, fitting, and statistical evaluation.

### Sample Structural Layout of a ROOT File
```text
MyData.root (The main file container)
└── Directory: Analysis
    ├── TTree: EventTree
    │   ├── TBranch: RunNumber (Stores integer run IDs)
    │   ├── TBranch: EventNumber (Stores integer event IDs)
    │   └── TBranch: ParticleMomenta (Stores vector arrays of particle paths)
    ├── TH1F: EnergyDistribution (A 1D histogram of energy counts)
    └── TCanvas: SummaryPlot (A saved visual canvas layout)
```

---
# 💻 Part 2: Interactive Tutorials & Lab Exercises
Work through the following macros in numerical order. All related data inputs, scripts, and plots are located within your cloned workspace directory.

**Instructions for Students:** Run each tutorial script using the execution blocks below. Review the code, inspect the interactive outputs, and attempt the critical thinking questions embedded at the end of each macro file.

### 📝 Tutorial 1: Random Numbers
Explore how ROOT processes pseudo-random generation algorithms via the command line.

In [ ]:
# Change working directory directly to the Exercises folder
os.chdir("/content/root-data-analysis-workshop_-LHC_School_2026-/Exercises")
print("Changed to Exercises/ Directory Successfully!!")

In [ ]:
import os

repo_dir = "/content/root-data-analysis-workshop_-LHC_School_2026-/Exercises"

# Search for tutorial01_random_numbers.C inside the cloned repo
macro_path = None
for root, dirs, files in os.walk(repo_dir):
    if "tutorial01_random_numbers.C" in files:
        macro_path = os.path.join(root, "tutorial01_random_numbers.C")
        break

if macro_path:
    print(f"✅ Found macro at: {macro_path}")
else:
    print("❌ Macro file not found. Listing top-level directory contents:")
    print(os.listdir(repo_dir) if os.path.exists(repo_dir) else "Repo folder missing.")

In [ ]:
!root -l -q -b tutorial01_random_numbers.C

### 📝 Tutorial 2: Creating Your First Histogram
Learn to instantiate a 1D histogram container (`TH1F`) and populate it with data parameters.

In [ ]:
!root -l -q -b tutorial02_first_histogram.C

### 📝 Tutorial 3: First Function Plot
Render mathematical formulas directly into continuous coordinate spaces using the `TF1` class functions.

In [ ]:
!root -l -q -b tutorial03_first_function_plot.C

### 📝 Tutorial 4: Loading Histograms from External Data Files
Read custom numerical measurements out of plain text configurations (`tutorial04_data.txt`) to build statistical histograms.

In [ ]:
!root -l -q -b tutorial04_histogram_from_file.C

### 📝 Tutorial 5: Basic Graph Generation from File
Map discrete coordinate paths out of text files (`tutorial05_data.txt`) using the raw `TGraph` rendering engine.

In [ ]:
!root -l -q -b tutorial05_graph_from_file.C

### 📝 Tutorial 6: Graphs with Experimental Error Measurements
Incorporate physical measurement uncertainties along both axes using the advanced `TGraphErrors` structure.

In [ ]:
!root -l -q -b tutorial06_graph_with_errors.C

### 📝 Tutorial 7: Generating Complex Gaussian Samples
Construct custom synthetic physics events by streaming pure Gaussian distributions into standard text file arrays.

In [ ]:
!root -l -q -b tutorial07_generate_gaussian_data.C

### 📝 Tutorial 8: Fitting Gaussian Parameters
Perform algorithmic chi-square regression fitting configurations across your collected dataset points.

In [ ]:
!root -l -q -b tutorial08_fit_gaussian_data.C

### 📝 Tutorial 9: Serializing and Saving Fits to `.root` Files
Save the fitted model distributions directly to compressed physical binary files (`tutorial09_gaussian_fit.root`).

In [ ]:
!root -l -q -b tutorial09_save_fit_to_root_file.C

### 📝 Tutorial 10: Re-opening and Inspecting Saved `.root` Files
Validate your storage pipelines by pulling preserved structures from disk back into memory blocks.

In [ ]:
!root -l -q -b tutorial10_open_root_file.C

In [ ]:
# Change working directory directly to the Assignments folder
os.chdir("/content/root-data-analysis-workshop_-LHC_School_2026-/Assignments")
print("Changed to Assignments/ Directory Successfully!!")

---
# 🎓 Part 3: Workshop Assignments

### 📂 Assignment 1: Statistical Binning Analysis
Objective: Use a control Gaussian sample to check and evaluate several histogram bin spacing configurations. Detail your observations regarding data resolution levels, bin item fluctuations, and whether the primary fitted parameters remain stable under structural mutations.

In [ ]:
# Execute starter macro for Assignment 1
!root -l -q -b assignment01_binning_study.C

### 📂 Assignment 2: Regression Model Selection
Objective: The provided starter macro builds a standard Gaussian event signal sitting on top of an invariant flat uniform background. A pure Gaussian-only fit calculation is therefore incomplete. Construct at least three unique composite candidate algebraic functions, evaluate their reduced chi-square results, and justify your final preferred model option.

In [ ]:
# Execute starter macro for Assignment 2
!root -l -q -b assignment02_model_selection.C